In [1]:
import warnings
warnings.filterwarnings('ignore')

# 주요 LLM 공급자 및 RAG 답변 평가

# 환경 설정

## 라이브러리 설치

!pip install "langchain>=0.3.0,<0.4.0" "langchain-core>=0.3.0,<0.4.0" "langchain-community>=0.3.0,<0.4.0" "langchain-text-splitters>=0.3.0,<0.4.0" "langchain-experimental>=0.3.0,<0.4.0" "langchain-openai" "langchain-anthropic" "langchain-chroma" "langchain-huggingface" "langchain-ollama" "langchain-google-genai" "langchain-groq" "pydantic>=2.7.0,<3.0.0" "anthropic>=0.30.0" google-genai groq krag kiwipiepy rank_bm25 jq sentence-transformers

`langchain-anthropic`: Anthropic(Claude), 코딩, 논리적 추론, 긴 문맥 처리에 강점이 있는 모델을 사용하기 위해 import 한다.  
`langchain-google-genai`: Google(Gemini), 구글의 최신 모델인 Gemini Pro, Flash 등을 사용하기 위해 import 한다.  
`langchain-groq`: 초고속 추론 엔진인 Groq을 통해 Llama 3, Mixtral 등을 사용하기 위해 import 한다.

## 기본 라이브러리

In [2]:
import os, json, re
from glob import glob
from pprint import pprint
import numpy as np
import pandas as pd

from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import JSONLoader
from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
# from langchain_community.embeddings import HuggingFaceEmbeddings # langchain-core 0.3.86일 경우 import 방법
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

## .env 환경 변수

In [3]:
from dotenv import load_dotenv
load_dotenv()

print('LangSmith 추적 여부:', os.getenv('LANGCHAIN_TRACING_V2'))

LangSmith 추적 여부: false


# 실습 환경 준비 및 검색기 초기화

# 데이터를 읽어온다.

## JSONL 파일 읽기

JSONL(JSON Line) 형식의 데이터를 읽어서, LangChain의 표준으로 사용하는 Document 객체 형태로 변환한다.

LangChain의 가장 기본적인 데이터 단위인 본문(page_content)과 부가 정보(metadata)를 묶어서 관리하는 Document를 import 한다.  
`from langchain_core.documents import Document`  
JSON 또는 JSONL 형식의 문자열을 파이썬에서 처리할 수 있도록 리스트나 딕셔너리 형태로 변환하거나 그 반대 작업을 하기위해 import 한다.  
`import json`

In [4]:
final_docs = []

# JSONL 파일을 읽기 위해서 final_docs.jsonl 파일을 읽어들인다.
# './data' 폴더의 'final_docs.jsonl' 파일을 바이너리 형태의 입력용으로 open 한다.
# 파일 모드는 인코딩 문제를 방지하기 위해 'rb'는 Read Binary의 약자인 '읽기 전용' 및 '바이너리 모드'로 연다.
# with 구문을 사용했으므로 with 블록의 모든 작업이 완료되면 파일이 자동으로 안전하게 닫힌다. 입력용 파일을 닫는 이유는 메모리를 효율적으로 사용하기 위해서이다.
with open('./data/final_docs.jsonl', 'rb') as file:
    for line in file:
        # json 라이브러리의 loads() 메소드는 인수로 지정된 문자열을 리스트나 딕셔너리 형태로 변환한다.
        item = json.loads(line)
        # 딕셔너리에서 'page_content'와 'metadata'를 꺼내서 Document 객체를 만들어 final_docs 리스트에 추가한다.
        doc = Document(page_content=item['page_content'], metadata=item['metadata'])
        final_docs.append(doc)

print(len(final_docs))
final_docs

6


[Document(metadata={'source': './data\\리비안_KR.txt', 'doc_id': 0}, page_content='리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다\n2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다\n주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'),
 Document(metadata={'source': './data\\리비안_KR.txt', 'doc_id': 1}, page_content='리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다\n이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'),
 Document(metadata={'source': './data\\리비안_KR.txt', 'doc_id': 2}, page_content='리비안은 디젤 하이브리드 버전, 브라질 원메이크 시리즈를 위한 R1 GT 레이싱 버전, 4도어 세단 및 크로스오버 등 다양한 버전을 고려했습니다\n2011년에 프로토타입 해치백도 공개되었지만, R1과의 관계는 불명확합니다\n리비안은 2021년 10월 첫 번째 양산 차량인 R1T 트럭을 고객에게 인도하기 시작했습니다\n\n(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)'),
 Document(metadata={'source': './data\\테슬라_KR.txt', 'doc_id': 3}, page_content='테슬라(Tesla, Inc\n)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니

## 엑셀 파일 읽기

판다스를 사용해서 엑셀 파일을 불러온다.

`import pandas as pd`

In [5]:
# 판다스가 제공하는 read_excel() 메소드로 엑셀 파일을 읽어온다. openpyxl 라이브러리가 설치되어있어야 정상적으로 동작된다.
df_qa_test = pd.read_excel('./data/qa_test.xlsx')
df_qa_test.head(6)

,context,source,doc_id,question,answer
0,"['.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2...",['data/리비안_KR.txt'],['0'],리비안의 초기 모델은 무엇인가요?,리비안의 초기 모델은 스포츠카 R1입니다.
1,"['.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2...",['data/리비안_KR.txt'],['0'],R1의 좌석 구성은 어떻게 되나요?,R1은 2+2 좌석 구성입니다.
2,"['.\n\n리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2...",['data/리비안_KR.txt'],['0'],R1은 어떤 구조를 특징으로 하나요?,R1은 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 합니다.
3,"['테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전...",['data/테슬라_KR.txt'],['1'],테슬라는 어디에 본사를 두고 있나요?,테슬라는 텍사스주 오스틴에 본사를 두고 있습니다.
4,"['테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전...",['data/테슬라_KR.txt'],['1'],테슬라는 언제 설립되었나요?,테슬라는 2003년에 설립되었습니다.
5,"['테슬라(Tesla, Inc.)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전...",['data/테슬라_KR.txt'],['1'],테슬라의 공동 창립자는 누구인가요?,테슬라의 공동 창립자는 마틴 에버하드와 마크 타페닝입니다.


# 검색기를 정의한다.

## 벡터검색기 - 시맨틱 검색, 맥락의 의미

임베딩 기술을 사용하여 텍스트의 의미를 숫자로 변환하고, 이를 바탕으로 질문과 가장 유사한 답변을 찾아내는 벡터검색기를 정의한다.

오픈소스 벡터 데이터이터베이스로 텍스트를 숫자로 바꾼 데이터를 저장하고 빠르게 검색하기 위해 Chroma를 import 한다.  
`from langchain_chroma import Chroma`  
오픈소스 모델 허브인 Hugging Face에서 제공하는 임베딩 모델을 사용하기 위해서 HuggingFaceEmbeddings를 import 한다.  
`from langchain_huggingface.embeddings import HuggingFaceEmbeddings`

벡터저장소에 사용한 임베딩 모델을 설정한다.

In [6]:
# HuggingFaceEmbeddings 클래스의 생성자로 임베딩 모델 이름을 넘겨서 임베딩 모델을 설정한다.
# 임베딩이란 컴퓨터 문장을 이해할 수 있는 숫자들로 바꾸는 과정이다.
embeddings_model = HuggingFaceEmbeddings(model='BAAI/bge-m3')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

벡터저장소를 로드한다.

In [7]:
# Chroma 클래스의 생성자로 임베딩 모델, 테이블 이름, 저장된 폴더를 넘겨서 벡터스토어를 로드한다.
chroma_db = Chroma(
    collection_name='hf_bge_m3',
    embedding_function=embeddings_model,
    persist_directory='./chroma_db',
)

벡터검색기를 만든다.

In [8]:
# as_retriever() 메소드로 질문과 유사도를 계산해서 의미적으로 가까운 가져올 문서 개수를 넘겨서 벡터검색기를 만든다.
chroma_k_retriever = chroma_db.as_retriever(search_kwargs={'k': 2})

벡터검색기를 실행한다.

In [9]:
query = '테슬라의 회장은 누구인가요?'
# invoke() 메소드로 쿼리를 넘겨서 거리가 가장 가까운 문서를 찾아 반환한다.
retriever_docs = chroma_k_retriever.invoke(query)

In [10]:
print(f'쿼리: {query}')
print('검색 결과')
for doc in retriever_docs:
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 100)

쿼리: 테슬라의 회장은 누구인가요?
검색 결과
테슬라(Tesla, Inc
)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다
2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'source': './data\\테슬라_KR.txt', 'doc_id': 3}
----------------------------------------------------------------------------------------------------
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다
회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다
테슬라는 2010년 6월 나스닥에 상장되었습니다
2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37
65% 증가했습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'doc_id': 4, 'source': './data\\테슬라_KR.txt'}
----------------------------------------------------------------------------------------------------


## BM25검색기 - 키워드 기반, 단어의 일치

한국어 형태소 분석기인 Kiwi를 활용하여, 텍스트의 키워드가 얼마나 일치하는지를 기준으로 문서를 찾아주는 KiWiBM25 검색기를 정의한다.

한국어 문장을 형태소 단위로 나눠주는 도구인 KiwiTokenizer를 import 한다.  
한국어 형태소 분석기인 Kiwi를 활용해서 관련 문서를 찾아주는 KiWiBM25RetrieverWithScore를 import 한다.

In [11]:
from krag.tokenizers import KiwiTokenizer
from krag.retrievers import KiWiBM25RetrieverWithScore

한국어를 분석에 사용할 토크나이저를 설정한다.

In [12]:
# KiwiTokenizer 클래스의 생성자로 토크나이저 모델과 오타 교정 여부를 넘겨서 토크나이저를 설정한다.
kiwi_tokenizer = KiwiTokenizer(
    # krag에서 가장 권장되는 모델인 특정 단어 뒤에 어떤 형태소가 나올 확률이 높은지 계산해서 문맥을 파악하는 knlm을 지정한다.
    model_type='knlm',
    # 형태소 분석시 일반적인 수준의 오타를 처리하는 'basic'을 지정한다.
    typos='basic'
)

KiWiBM25 검색기를 만든다.

In [13]:
# KiWiBM25RetrieverWithScore 클래스의 생성자로 토큰화할 문서, kiwi 토크나이저, 가져올 문서 개수, 검색 점수 임계값을 넘겨서 KiWiBM25 검색기를 만든다.
kiwibm25_k_retriever = KiWiBM25RetrieverWithScore(
    # KiWiBM25 검색기를 만들기 위해서 토큰화 할 문서 리스트를 지정한다.
    documents=final_docs,
    # 형태소 분석에 사용할 Kiwi 토크나이저를 지정한다.
    kiwi_tokenizer=kiwi_tokenizer,
    # 검색해서 가장 관련성이 높은 점수 순서로 가져올 문서의 개수를 지정한다.
    k=2,
    # 검색 점수의 임계값을 지정한다. 0.0으로 지정하면 점수에 상관없이 모든 결과를 가져온다.
    threshold=0.0
)

KiWiBM25 검색기를 실행한다.

In [14]:
query = '테슬라의 회장은 누구인가요?'
retriever_docs = kiwibm25_k_retriever.invoke(query)

In [15]:
print(f'쿼리: {query}')
print('검색 결과')
for doc in retriever_docs:
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 100)

쿼리: 테슬라의 회장은 누구인가요?
검색 결과
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다
회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다
테슬라는 2010년 6월 나스닥에 상장되었습니다
2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37
65% 증가했습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'source': './data\\테슬라_KR.txt', 'doc_id': 4, 'bm25_score': 2.1003500487956686}
----------------------------------------------------------------------------------------------------
리비안의 초기 모델은 스포츠카 R1(원래 이름은 Avera)로, 2+2 좌석의 미드 엔진 하이브리드 쿠페로 피터 스티븐스가 디자인했습니다
이 차는 쉽게 교체 가능한 본체 패널을 갖춘 모듈식 캡슐 구조를 특징으로 하며, 2013년 말에서 2014년 초 사이에 생산이 예상되었습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'source': './data\\리비안_KR.txt', 'doc_id': 1, 'bm25_score': 0.8556885950901274}
----------------------------------------------------------------------------------------------------


## Ensemble 검색기 - Hybrid Search

서로 다른 2개 이상의 검색기(Chroma(의미 기반), KiwiBM25(키워드 기반))를 하나로 합쳐서 각각의 장점만 취하는 Ensemble 검색기를 정의한다.

여러 개의 검색 알고리즘을 결합하여 더 나은 검색 결과를 만들어내는 EnsembleRetriever를 import 한다.

In [16]:
from langchain.retrievers import EnsembleRetriever

Ensemble 검색기를 만든다.

In [17]:
# Chroma 벡터저장소나 KiwiBM25 저장소를 새로 만들지 않고 기존 저장소의 검색기의 상위 k을 아래와 같이 변경할 수 있다.
chroma_k_retriever.search_kwargs['k'] = 4
kiwibm25_k_retriever.k = 4

In [18]:
# EnsembleRetriever 클래스의 생성자로 사용할 서로 다른 검색기 목록과 각 검색기의 비중을 넘겨서 Ensemble 검색기를 만든다.
ensemble_k_retriever = EnsembleRetriever(
    # 사용할 서로 다른 검색기 목록을 리스트 형태로 지정한다.
    retrievers=[chroma_k_retriever, kiwibm25_k_retriever],
    # 각 검색기별 가중치를 리스트 형태로 지정한다. 이 수치를 조절해서 검색 성능을 튜닝할 수 있다.
    weights=[0.5, 0.5]
)

Ensemble 검색기를 실행한다.

In [19]:
query = '테슬라의 회장은 누구인가요?'
retriever_docs = ensemble_k_retriever.invoke(query)

In [20]:
print(f'쿼리: {query}')
print('검색 결과')
for doc in retriever_docs:
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 100)

쿼리: 테슬라의 회장은 누구인가요?
검색 결과
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다
회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다
테슬라는 2010년 6월 나스닥에 상장되었습니다
2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37
65% 증가했습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'doc_id': 4, 'source': './data\\테슬라_KR.txt'}
----------------------------------------------------------------------------------------------------
2012년부터 2023년 3분기까지 테슬라의 전 세계 누적 판매량은 4,962,975대를 초과했습니다
SMT Packaging에 따르면, 2023년 테슬라의 판매량은 전 세계 전기차 시장의 약 12
9%를 차지했습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'source': './data\\테슬라_KR.txt', 'doc_id': 5}
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다
2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다
주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'doc_id': 0, 'source': './data\\리비안_KR.txt'}
-------------------------------------------------------------------

## CrossEncoder 알고리즘에 기반하여 순위를 다시 매긴다. - Rerank

Hugging Face에서 제공하는 CrossEncoder 모델을 불러와서 재순위화 하기 위해서 HuggingFaceCrossEncoder를 import 한다.  
LangChain에서 RAG 시스템의 성능 향상시키기 위해 사용하는 문서 재순위화 하기 위해서 CrossEncoderReranker import 한다.  
LangChain에서 기본 검색기에서 가져온 문서 중 질문과 진짜 관련된 내용만 압축, 선별해서 LLM에 전달하기 위해서 ContextualCompressionRetriever import 한다.

In [21]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever

재순위화에 사용할 재순위화 모델을 불러온다.

In [22]:
# HuggingFaceCrossEncoder 클래스의 생성자로 재순위화에 사용할 모델 이름을 넘겨서 재순위화 모델을 설정한다. 
model = HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-v2-m3')

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

불러온 재순위화 모델을 이용해서 실제로 재순위화 작업을 하는 모델을 만든다.

In [23]:
# CrossEncoderReranker 클래스의 생성자로 재순위화에 사용할 모델과 점수에 따라 가져올 문서 개수를 넘겨서 질문과 관련된 내용만 압축, 선별하는 모델을 만든다.
re_ranker = CrossEncoderReranker(model=model, top_n=3)

기본 검색기와 재순위화 검색기를 이용하는 최종 검색기를 만든다.

In [24]:
# ContextualCompressionRetriever 클래스의 생성자로 기본 검색기와 기본 압축기를 넘겨서 최종 검색기를 만든다.
cross_encoder_reranker_retriever = ContextualCompressionRetriever(
    # 기본 검색기를 Chroma(의미 기반), KiwiBM25(키워드 기반)를 하나로 합쳐서 각각의 장점만 취하는 Ensemble 검색기를 지정한다.
    base_retriever=ensemble_k_retriever,
    # 기본 검색기에서 얻어온 문서를 재순위화를 하는 모델을 지정한다.
    base_compressor=re_ranker
)

ensemble_k_retriever 검색기에서 얻어온 문서의 재순위화를 실행한다.

In [25]:
query = '테슬라의 회장은 누구인가요?'
retriever_docs = cross_encoder_reranker_retriever.invoke(query)

In [26]:
print(f'쿼리: {query}')
print('검색 결과')
for doc in retriever_docs:
    print(doc.page_content)
    print(doc.metadata)
    print('-' * 100)

쿼리: 테슬라의 회장은 누구인가요?
검색 결과
테슬라(Tesla, Inc
)는 텍사스주 오스틴에 본사를 둔 미국의 대표적인 전기차 제조업체입니다
2003년 마틴 에버하드(CEO)와 마크 타페닝(CFO)에 의해 설립된 테슬라는 2004년 페이팔과 Zip2의 공동 창업자인 일론 머스크의 참여로 큰 전환점을 맞았습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'doc_id': 3, 'source': './data\\테슬라_KR.txt'}
----------------------------------------------------------------------------------------------------
머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다
회사 이름은 유명한 물리학자이자 전기공학자인 니콜라 테슬라의 이름을 따서 지어졌습니다
테슬라는 2010년 6월 나스닥에 상장되었습니다
2023년 테슬라는 1,808,581대의 차량을 판매하여 2022년에 비해 37
65% 증가했습니다

(참고: 이 문서는 테슬라에 대한 정보를 담고 있습니다.)
{'doc_id': 4, 'source': './data\\테슬라_KR.txt'}
----------------------------------------------------------------------------------------------------
리비안은 MIT 박사 출신 RJ 스카린지가 2009년에 설립한 혁신적인 미국 전기차 제조업체입니다
2011년부터 자율 전기차에 집중한 리비안은 2015년 대규모 투자를 통해 크게 성장하며 미시간과 베이 지역에 연구소를 설립했습니다
주요 공급업체와의 접근성을 높이기 위해 본사를 미시간주 리보니아로 이전했습니다

(참고: 이 문서는 리비안에 대한 정보를 담고 있습니다.)
{'doc_id': 0, 'source': './data\\리비안_KR.txt'}
---------------------------------------

# 주요 LLM 공급자

# OpenAI

OpenAI의 GPT 모델은 현재 가장 널리 알려진 LLM 중 하나로 GPT-3.5, GPT-4, GPT-5 등 다양한 버전을 제공한다.  
텍스트 생성, 번역, 요약 등 다양한 작업에서 우수한 성능을 보이고 잘 문서화된 API와 다양한 라이브러리를 지원한다.

`05_고급검색기법` 파일의 `RAG 파이프라인으로 적용하기` 부분을 함수 형태로 만들어 실행한다.

LLM에게 전달할 메시지를 정의하기 위해 ChatPromptTemplate를 import 한다.  
`from langchain_core.prompts import ChatPromptTemplate`  
LangChain에서 제공하는 OpenAI의 대화형 AI을 활용하기 위해 ChatOpenAI를 import 한다.  
`from langchain_openai import ChatOpenAI`  
LLM이 응답한 객체에서 문자열만 뽑아내기 위해 StrOutputParser를 import 한다.  
`from langchain_core.output_parsers import StrOutputParser`

In [27]:
# 여기서는 질문을 가공 없이 그대로 프롬프트에 넘기기 위해서 RunnablePassthrough를 import 한다.
from langchain_core.runnables import RunnablePassthrough

검색기와 LLM 모델을 인수로 넘겨받아 프롬프트를 만들고 LLM 모델에 전달해서 LLM 모델이 응답한 결과를 문자열만 얻어오는 RAG chain을 리턴하는 함수를 정의한다.

In [28]:
def create_rag_chain(retriever, llm):
    # 프롬프트 템플릿
    template = '''
    당신은 유능한 나만의 인공지능 비서입니다.
    당신이 수행할 과제는 제공된 문맥(context)을 바탕으로 다음 질문에 답변하는 것입니다.
    다음 문맥(context)에만 근거하여 질문에 답변하고 외부 정보나 지식은 절대로 사용하지 마세요.
    만약 문맥(context)이 질문과 관련이 없다면, '답변에 필요한 근거를 찾지 못했습니다.'라고 답변하세요.

    [문맥(context)]
    {context}

    [질문]
    {question}

    [답변]
    '''
    prompt = ChatPromptTemplate.from_template(template=template)
    
    # 검색기가 검색한 문서들에서 본문 부분만 꺼내서 하나의 긴 문자열로 연결하는 함수
    def format_docs(docs):
        return '\n\n'.join([doc.page_content for doc in docs])
    
    # RAG Chain을 만들어서 리턴시킨다.
    return {'context': retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser()

In [29]:
# LLM 모델 정의
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.0)

# 검색기와 LLM 모델을 넘겨서 RAG chain을 얻어오는 함수 실행
openai_rag_chain = create_rag_chain(cross_encoder_reranker_retriever, llm)

# RAG chain 실행
question = '테슬라의 회장은 누구인가요?'
answer = openai_rag_chain.invoke(question)

In [30]:
print(f'질문: {question}')
print(f'답변: {answer}')

질문: 테슬라의 회장은 누구인가요?
답변: 일론 머스크입니다.


# Anthropic Claude

`https://platform.claude.com/`에 접속해서 회원 가입 후 API key를 발급받고 .env 파일에 `ANTHROPIC_API_KEY`를 등록한다.

현재 사용할 수 있는 모델은 아래 링크를 통해서 확인할 수 있다.

`https://platform.claude.com/docs/ko/about-claude/model-deprecations`

<img src="./claude.png" width="1200" align="left" />

클로드 API 모델 중 사용 목적과 성능 대비 비용(가성비) 관점에서 가장 우수한 모델은 claude-sonnet-5와 claude-haiku-4-5-20251001 두 가지이다.

`claude-sonnet-5`: 대부분의 개발 및 복잡한 작업(가장 균형 잡힌 가성비)  
`claude-haiku-4-5-20251001`: 단순 반복 업무 및 대량 단순 처리(비용 최소화)  

LangChain에서 LLM 모델인 Claude 시리즈를 사용하기 위해 ChatAnthropic를 import 한다.

In [31]:
from langchain_anthropic import ChatAnthropic

Claude 모델을 생성한다.

In [32]:
llm = ChatAnthropic(
    # 사용할 Anthropic Claude의 모델 이름을 지정한다.
    model_name='claude-haiku-4-5-20251001',
    # 모델이 답변을 생성할 때의 창의성(자유도)과 무작위성을 조절한다.
    temperature=0.0,
    # 모델이 한 번의 응답에서 생성할 수 있는 최대 토큰 수를 지정한다.
    max_tokens_to_sample=200
)

create_rag_chain 함수로 검색기와 LLM 모델을 넘겨서 RAG chain을 리턴받는다.

In [33]:
anthropic_rag_chain = create_rag_chain(cross_encoder_reranker_retriever, llm)

RAG chain을 실행한다.

In [34]:
question = '테슬라의 회장은 누구인가요?'
answer = anthropic_rag_chain.invoke(question)

In [35]:
print(f'질문: {question}')
print(f'답변: {answer}')

질문: 테슬라의 회장은 누구인가요?
답변: 제공된 문맥에 따르면, 테슬라의 회장은 **일론 머스크(Elon Musk)**입니다.

문맥에서 "머스크는 최대 주주이자 회장으로서 회사를 현재의 성공으로 이끌었습니다"라고 명시되어 있습니다.


# Google Gemini

`https://ai.google.dev/`에 접속해서 회원 가입 후 API key를 발급받고 .env 파일에 `GOOGLE_API_KEY`를 등록한다.

현재 사용할 수 있는 모델은 아래 링크를 통해서 확인할 수 있다.

`https://aistudio.google.com/docs/models`

<img src="./gemini.png" width="1200" align="left" />

`gemini-2.5-flash`: Gemini에서 최고 가성비 모델

LangChain에서 LLM 모델인 Gemini 시리즈를 사용하기 위해 ChatGoogleGenerativeAI를 import 한다.

In [36]:
from langchain_google_genai import ChatGoogleGenerativeAI

Claude 모델을 생성한다.

In [37]:
llm = ChatGoogleGenerativeAI(
    # 사용할 Gemini 모델 이름을 지정한다.
    model='gemini-2.5-flash',
    # 모델이 답변을 생성할 때의 창의성(자유도)과 무작위성을 조절한다.
    temperature=0.0,
    # 모델이 한 번의 응답에서 생성할 수 있는 최대 토큰 수를 지정한다.
    max_tokens=200
)

create_rag_chain 함수로 검색기와 LLM 모델을 넘겨서 RAG chain을 리턴받는다.

In [38]:
google_genai_rag_chain = create_rag_chain(cross_encoder_reranker_retriever, llm)

RAG chain을 실행한다.

In [39]:
question = '테슬라의 회장은 누구인가요?'
answer = google_genai_rag_chain.invoke(question)

In [40]:
print(f'질문: {question}')
print(f'답변: {answer}')

질문: 테슬라의 회장은 누구인가요?
답변: 테슬라의 회장은 일론 머스크입니다.


# Ollama

`ollama list` 명령로 컴퓨터에 설치된 모델 목록을 확인하고 다운로드된 모델이 없으면 `ollama pull 모델이름` 명령으로 사용할 모델을 다운로드 받아야 한다.

<img src="./ollama_5.png" width="1200" align="left" />

In [41]:
# !ollama list
# !ollama pull llama3.1

LangChain에서 LLM 모델인 Ollama 시리즈를 사용하기 위해 ChatOllama를 import 한다.

In [42]:
from langchain_ollama import ChatOllama

Ollama 모델을 생성한다.

In [43]:
llm = ChatOllama(
    # 사용할 Ollama 모델 이름을 지정한다.
    model='llama3.1',
    # 모델이 답변을 생성할 때의 창의성(자유도)과 무작위성을 조절한다.
    temperature=0.0,
    # 모델이 한 번의 응답에서 생성할 수 있는 최대 토큰 수를 지정한다.
    num_predict=200
)

create_rag_chain 함수로 검색기와 LLM 모델을 넘겨서 RAG chain을 리턴받는다.

In [44]:
ollama_rag_chain = create_rag_chain(cross_encoder_reranker_retriever, llm)

RAG chain을 실행한다.

In [45]:
question = '테슬라의 회장은 누구인가요?'
answer = ollama_rag_chain.invoke(question)

In [46]:
print(f'질문: {question}')
print(f'답변: {answer}')

질문: 테슬라의 회장은 누구인가요?
답변: 일론 머스크입니다.


# Groq

`https://groq.com/`에 접속해서 회원 가입 후 API key를 발급받고 .env 파일에 GROQ_API_KEY를 등록한다.

현재 계정이 사용 가능한 Groq 모델 목록을 확인한다.

In [47]:
from groq import Groq

In [48]:
groq = Groq(api_key=os.environ.get('GROQ_API_KEY'))
models = groq.models.list()

print('==== 현재 사용 가능한 Groq 모델 목록 ====')
for model in models.data:
    print(model.id)

==== 현재 사용 가능한 Groq 모델 목록 ====
openai/gpt-oss-120b
whisper-large-v3-turbo
canopylabs/orpheus-v1-english
whisper-large-v3
openai/gpt-oss-20b
allam-2-7b
openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-arabic-saudi
qwen/qwen3.8-27b


allam-2-7b - 사용가능, 바보  
`openai/gpt-oss-safeguard-20b`  
`qwen/qwen3.8-27b`  
meta-llama/llama-prompt-guard-2-22m - 사용불가능  
whisper-large-v3-turbo - 사용불가능  
canopylabs/orpheus-arabic-saudi - 사용불가능   
whisper-large-v3 - 사용불가능   
meta-llama/llama-prompt-guard-2-86m - 사용가능, 바보   
`openai/gpt-oss-120b`  
canopylabs/orpheus-v1-english - 사용불가능    
`openai/gpt-oss-20b`

`openai/gpt-oss-120b`: Groq에서 최고 가성비 모델

LangChain에서 LLM 모델인 Groq 시리즈를 사용하기 위해 ChatGroq를 import 한다.

In [49]:
from langchain_groq import ChatGroq

Groq 모델을 생성한다.

In [50]:
llm = ChatGroq(
    # 사용할 Groq 모델 이름을 지정한다.
    model='qwen/qwen3.8-27b',
    # 모델이 답변을 생성할 때의 창의성(자유도)과 무작위성을 조절한다.
    temperature=0.0,
    # 모델이 한 번의 응답에서 생성할 수 있는 최대 토큰 수를 지정한다.
    max_tokens=200
)

create_rag_chain 함수로 검색기와 LLM 모델을 넘겨서 RAG chain을 리턴받는다.

In [51]:
groq_rag_chain = create_rag_chain(cross_encoder_reranker_retriever, llm)

RAG chain을 실행한다.

In [52]:
question = '테슬라의 회장은 누구인가요?'
answer = groq_rag_chain.invoke(question)

In [53]:
print(f'질문: {question}')
print(f'답변: {answer}')

질문: 테슬라의 회장은 누구인가요?
답변: 테슬라의 회장은 일론 머스크입니다.


# RAG 답변 평가 - 정량적 지표

# RAG 답변 평가 - LLM-as-judge